# Scheme Navigator — free IndicTrans2 pack generator
Run this on a Colab GPU. The cache is copied to Google Drive so disconnects are resumable.

**Before running:** request/accept access to `ai4bharat/indictrans2-en-indic-dist-200M` on Hugging Face, then create a **Read** access token from the same Hugging Face account. The token is entered into a hidden prompt below and is never saved in this notebook/repository.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/scheme-navigator
!git clone https://github.com/um26/scheme-navigator.git /content/scheme-navigator
%cd /content/scheme-navigator


In [ ]:
!pip -q install -U 'torch>=2.5' 'transformers>=4.51,<5' indictranstoolkit sentencepiece sacremoses accelerate huggingface_hub
!npm install --silent


## Authenticate to Hugging Face
The web page saying **granted access** is necessary, but Colab must also log in with a token from that **same account**. Paste a Hugging Face **Read** token into the hidden prompt. Do not paste the token into chat, notebook text, or GitHub.


In [ ]:
from getpass import getpass
from huggingface_hub import login, HfApi

MODEL_NAME = 'ai4bharat/indictrans2-en-indic-dist-200M'
HF_TOKEN = getpass('Paste your Hugging Face READ token (input is hidden): ')
login(token=HF_TOKEN, add_to_git_credential=False)

api = HfApi(token=HF_TOKEN)
me = api.whoami()
print(f"✅ Authenticated to Hugging Face as: {me.get('name') or me.get('fullname') or 'your account'}")
try:
    info = api.model_info(MODEL_NAME, token=HF_TOKEN)
    print(f"✅ Gated model access verified: {info.modelId}")
except Exception as exc:
    raise RuntimeError(
        'Hugging Face login worked, but this token/account still cannot access the IndicTrans2 model. '
        'Make sure the token was created from the SAME account that shows “granted access”, then rerun this cell.'
    ) from exc


In [ ]:
# Restore resumable cache from Drive, if one exists.
!mkdir -p .translation-work public/i18n/schemes
!cp -r /content/drive/MyDrive/scheme-navigator-i18n/cache .translation-work/ 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json public/i18n/generated-ui.json 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/generated.js lib/i18n/generated.js 2>/dev/null || true
!cp /content/drive/MyDrive/scheme-navigator-i18n/*.json.gz public/i18n/schemes/ 2>/dev/null || true


In [ ]:
# Start with a manageable batch. Change this after each completed run.
LOCALES = 'hi,mr,bn,gu'
!python scripts/translate-catalog.py --locales $LOCALES --batch-size 32


In [ ]:
# Persist progress + create an uploadable artifact only after the translation cell succeeds.
from pathlib import Path
packs = list(Path('public/i18n/schemes').glob('*.json.gz'))
if not packs:
    raise RuntimeError('No translation packs exist yet. Run the translation cell successfully before packaging.')

!mkdir -p /content/drive/MyDrive/scheme-navigator-i18n
!rm -rf /content/drive/MyDrive/scheme-navigator-i18n/cache
!cp -r .translation-work/cache /content/drive/MyDrive/scheme-navigator-i18n/cache
!cp public/i18n/generated-ui.json /content/drive/MyDrive/scheme-navigator-i18n/generated-ui.json
!cp lib/i18n/generated.js /content/drive/MyDrive/scheme-navigator-i18n/generated.js
!cp public/i18n/schemes/*.json.gz /content/drive/MyDrive/scheme-navigator-i18n/
!zip -q -r /content/scheme-navigator-translations.zip public/i18n lib/i18n/generated.js
print('✅ Translation packs:', ', '.join(p.name for p in packs))
print('Artifact: /content/scheme-navigator-translations.zip')
